In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv("cardekho_dataset.csv", index_col=0)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15411 entries, 0 to 19543
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   car_name           15411 non-null  object 
 1   brand              15411 non-null  object 
 2   model              15411 non-null  object 
 3   vehicle_age        15411 non-null  int64  
 4   km_driven          15411 non-null  int64  
 5   seller_type        15411 non-null  object 
 6   fuel_type          15411 non-null  object 
 7   transmission_type  15411 non-null  object 
 8   mileage            15411 non-null  float64
 9   engine             15411 non-null  int64  
 10  max_power          15411 non-null  float64
 11  seats              15411 non-null  int64  
 12  selling_price      15411 non-null  int64  
dtypes: float64(2), int64(5), object(6)
memory usage: 1.6+ MB


In [4]:
df.head()

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [5]:
df.drop(columns=['car_name', 'brand'], axis=1, inplace=True)
cat_features = [features for features in df.columns if df[features].dtype == 'O']
num_features = [features for features in df.columns if df[features].dtype != 'O']

In [6]:
df.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [7]:
X = df.drop(['selling_price'], axis=1)
y = df['selling_price']

In [8]:
from sklearn.preprocessing import LabelEncoder

X['model'] = LabelEncoder().fit_transform(X['model'])

In [9]:
X.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,7,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,54,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,118,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,7,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,38,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


In [10]:
num_features = X.select_dtypes(exclude='object').columns
ohc = ['seller_type', 'fuel_type', "transmission_type"]

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

process = ColumnTransformer([
    ("OneHotEncoder", OneHotEncoder(drop='first'), ohc),
    ("StandardScaler", StandardScaler(), num_features)
], remainder='passthrough')

In [11]:
X = process.fit_transform(X)

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [13]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor()
model.fit(X_train, y_train)
model.predict(X_train)

array([ 211690.        , 1286770.        ,  570056.66666667, ...,
        282018.33333333,  668486.66666667,  936830.        ],
      shape=(11558,))

In [14]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso(),
    "Decision Tree": DecisionTreeRegressor(),
    "Support Vector Machines": SVR(),
    "RandomForest": RandomForestRegressor()
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    model_train_r2 = r2_score(y_train, y_train_pred)
    model_train_mae = mean_absolute_error(y_train, y_train_pred)
    model_train_mse = mean_squared_error(y_train, y_train_pred)
    model_train_rmse = np.sqrt(model_train_mse)
    
    model_test_r2 = r2_score(y_test, y_test_pred)
    model_test_mae = mean_absolute_error(y_test, y_test_pred)
    model_test_mse = mean_squared_error(y_test, y_test_pred)
    model_test_rmse = np.sqrt(model_test_mse)

    print("-------", list(models.keys())[i], "-------")
    print("\n")          

    print('Model Performance for Training Set')
    print("- R2 Score: {:.4f}".format(model_train_r2))            
    print("- MAE: {:.4f}".format(model_train_mae))            
    print("- MSE: {:.4f}".format(model_train_mse))            
    print("- RMSE: {:.4f}".format(model_train_rmse))            
    print("\n")          

    print('Model Performance for Test Set')
    print("- R2 Score: {:.4f}".format(model_test_r2))            
    print("- MAE: {:.4f}".format(model_test_mae))            
    print("- MSE: {:.4f}".format(model_test_mse))            
    print("- RMSE: {:.4f}".format(model_test_rmse))  
    print("="*35)
    print("\n")          

------- Linear Regression -------


Model Performance for Training Set
- R2 Score: 0.6220
- MAE: 266675.1076
- MSE: 304874315292.8461
- RMSE: 552154.2495


Model Performance for Test Set
- R2 Score: 0.6525
- MAE: 284283.4460
- MSE: 270286925822.7529
- RMSE: 519891.2635


------- Ridge Regression -------


Model Performance for Training Set
- R2 Score: 0.6220
- MAE: 266635.3662
- MSE: 304875008766.8082
- RMSE: 552154.8775


Model Performance for Test Set
- R2 Score: 0.6525
- MAE: 284241.1129
- MSE: 270275613895.9654
- RMSE: 519880.3842


------- Lasso Regression -------


Model Performance for Training Set
- R2 Score: 0.6220
- MAE: 266674.0472
- MSE: 304874327628.1807
- RMSE: 552154.2607


Model Performance for Test Set
- R2 Score: 0.6525
- MAE: 284283.7890
- MSE: 270286207881.6533
- RMSE: 519890.5730


------- Decision Tree -------


Model Performance for Training Set
- R2 Score: 0.9995
- MAE: 4991.7517
- MSE: 426210181.9807
- RMSE: 20644.8585


Model Performance for Test Set
- R2 Scor